In [4]:
# -*- coding: utf-8 -*-
# Block 1: Imports (Unchanged)
import os
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2
try:
    from segmentation_models_pytorch.losses import DiceLoss, LovaszLoss # Keep Lovasz
except ImportError:
    print("Please install segmentation_models_pytorch: pip install segmentation-models-pytorch")
    raise
try:
    import timm
except ImportError:
    print("Please install timm: pip install timm")
    raise
import torch.nn.functional as F
from torch.optim.lr_scheduler import CosineAnnealingLR
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.colors as mcolors
import glob
from tqdm.auto import tqdm
import time
import warnings
import random # Needed for balancing
import math # Needed for FoV mask calculation
import shutil # For potentially cleaning directories


warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)


# Block 2: Configuration and Paths (MODIFIED FOR PATCHING - CORRECTED SIZE)

# --- Paths ---
# (Paths remain the same)
_segmentation_root_part1 = "/content/drive/MyDrive/IDRID/A.%20Segmentation/"
_segmentation_root_part2 = "A. Segmentation"
TRAIN_IMG_DIR = os.path.join(_segmentation_root_part1, _segmentation_root_part2, "1. Original Images", "a. Training Set")
TRAIN_MASK_BASE_DIR = os.path.join(_segmentation_root_part1, _segmentation_root_part2, "2. All Segmentation Groundtruths", "a. Training Set")
print(f"Using Original Train Img Path: {TRAIN_IMG_DIR}")
print(f"Using Original Train Mask Base Path: {TRAIN_MASK_BASE_DIR}")
if not os.path.isdir(TRAIN_IMG_DIR): raise FileNotFoundError(f"Train Image Directory not found: {TRAIN_IMG_DIR}")
if not os.path.isdir(TRAIN_MASK_BASE_DIR): raise FileNotFoundError(f"Train Mask Base Directory not found: {TRAIN_MASK_BASE_DIR}")

# --- Save Directories (MODIFIED FOR PATCHES & CORRECTED SIZE) ---
SAVE_DIR = "/content/drive/MyDrive/segmentation"
TASK_SUFFIX = "hard_exudates"
BASE_CONFIG_SUFFIX = "256_attn_swin_s_w16_patch"
MODEL_NAME = f"idrid_{TASK_SUFFIX}_{BASE_CONFIG_SUFFIX}_fulltrain_patch" # Added suffix for clarity
BEST_CHECKPOINT_PATH = os.path.join(SAVE_DIR, f"{MODEL_NAME}_best_model.pth")
FINAL_MODEL_PATH = os.path.join(SAVE_DIR, f"{MODEL_NAME}_final_model.pth")
TRAINING_PLOTS_PATH = os.path.join(SAVE_DIR, f"{MODEL_NAME}_training_plots.png")
VIS_DIR = os.path.join(SAVE_DIR, "visualizations_" + MODEL_NAME)
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(VIS_DIR, exist_ok=True)
print(f"MODEL_NAME set to: {MODEL_NAME}")
print(f"Saving checkpoints to: {BEST_CHECKPOINT_PATH}")

# --- Patch Directories (Unchanged definitions) ---
PATCH_SAVE_DIR_BASE = os.path.join(SAVE_DIR, "balanced_patches") # Store balanced (but not augmented) patches
PATCH_SAVE_IMG_DIR = os.path.join(PATCH_SAVE_DIR_BASE, "Images")
PATCH_SAVE_MASK_DIR_BASE = os.path.join(PATCH_SAVE_DIR_BASE, "Masks") # Base dir for masks

# --- Hyperparameters ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PATCH_SIZE = 256
STRIDE = 128 # Adjust stride if needed (e.g., 50% overlap for 384)
IMAGE_SIZE = PATCH_SIZE # Model input size = Patch size
# <<< SET BATCH SIZE APPROPRIATELY - START HIGHER >>>
BATCH_SIZE = 64 # START with 8 or 4 for 384x384. Reduce ONLY if OOM occurs.
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 1e-4
NUM_EPOCHS = 500 # Adjust as needed
PATIENCE = 400

# Encoder settings (keep consistent)
ENCODER = 'swinv2_small_window16_256' # This model expects 384x384 fine-tuned size
ENCODER_WEIGHTS = 'imagenet'
ACTIVATION = None
NUM_WORKERS = 2 # Keep 2 for Kaggle, or adjust based on environment
DEBUG_FORWARD_PASS = False

# --- !!! Hard Exudates Segmentation Setup (Unchanged) !!! ---
CLASS_NAMES = ['EX'] # Hard Exudates
MASK_FOLDER_NAME = '3. Hard Exudates' # <<< CHECK THIS FOLDER NAME <<<
MASK_SUBDIR_NAMES = { CLASS_NAMES[0]: MASK_FOLDER_NAME }
NUM_CLASSES = 1 # Binary
VIS_COLORS = ['yellow']

# --- Loss Function Weights (Unchanged) ---
BCE_WEIGHT = 0.5
DICE_WEIGHT = 0.5

# Block 3: Preprocessing & Augmentation Helpers (MODIFIED FOR NO OFFLINE AUG + ONLINE CLAHE)
import albumentations as A
import cv2
import os
import glob
from tqdm.auto import tqdm
import numpy as np
from PIL import Image
from albumentations.pytorch import ToTensorV2
import math # Needed for FoV mask calculation
import random # Needed for balancing
import shutil # For cleaning dirs

# --- Normalization constants (Unchanged) ---
_NORM_MEAN = (0.485, 0.456, 0.406)
_NORM_STD = (0.229, 0.224, 0.225)

# --- Preprocessing Helper Functions (Keep definitions, but CLAHE/Gamma not called offline) ---
# apply_clahe_preprocess (Not used directly, using A.CLAHE instead)
def apply_clahe_preprocess(image, clip_limit_l=2.0, tile_grid_size=(8, 8), **kwargs):
    if not isinstance(image, np.ndarray): image = np.array(image)
    if image.ndim == 2: image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
    if image.shape[-1] != 3: raise ValueError("Input image must be RGB.")
    if image.dtype != np.uint8: image = np.clip(image, 0, 255).astype(np.uint8)
    lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB); l, a, b = cv2.split(lab)
    clahe_l = cv2.createCLAHE(clipLimit=clip_limit_l, tileGridSize=tile_grid_size)
    l_clahe = clahe_l.apply(l); lab_clahe = cv2.merge((l_clahe, a, b))
    clahe_img_rgb = cv2.cvtColor(lab_clahe, cv2.COLOR_LAB2RGB)
    return clahe_img_rgb

# adaptive_gamma_correction (Not used)
def adaptive_gamma_correction(image, gamma_scale=1.2, **kwargs):
    if not isinstance(image, np.ndarray): image = np.array(image)
    if image.ndim == 2: image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
    if image.shape[-1] != 3: raise ValueError("Input image must be RGB.")
    if image.dtype != np.uint8: image = np.clip(image, 0, 255).astype(np.uint8)
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY); mean_intensity = np.mean(gray) / 255.0
    gamma = np.clip(1.0 + (gamma_scale - 1.0) * (0.5 - mean_intensity), 0.4, 2.5)
    if mean_intensity <= 0: gamma = 1.0
    lookup_table = np.array([np.clip(((i / 255.0) ** gamma) * 255, 0, 255) for i in range(256)]).astype(np.uint8)
    return cv2.LUT(image, lookup_table)

# Field of View (FoV) Mask Function (Unchanged, Used before patching)
def create_fov_mask(image, threshold=15, fill_color=255):
    if image is None: return None
    if image.ndim == 3: gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    else: gray = image
    _, thresh = cv2.threshold(gray, threshold, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours: return np.full(gray.shape, fill_color, dtype=np.uint8) # Return full mask if no contour
    largest_contour = max(contours, key=cv2.contourArea)
    fov_mask = np.zeros_like(gray, dtype=np.uint8)
    cv2.drawContours(fov_mask, [largest_contour], -1, fill_color, thickness=cv2.FILLED)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(5,5))
    fov_mask = cv2.morphologyEx(fov_mask, cv2.MORPH_OPEN, kernel)
    fov_mask = cv2.morphologyEx(fov_mask, cv2.MORPH_CLOSE, kernel)
    return fov_mask


# --- Function to Generate Balanced Patches (Unchanged) ---
def generate_balanced_patches(
    image_dir, mask_base_dir, target_class_name, target_subdir_name,
    patch_save_img_dir, patch_save_mask_dir_base, # Note: mask base dir for output
    patch_size, stride):
    """
    Generates overlapping patches, balances lesion/non-lesion patches,
    and saves them to specified directories. Applies FoV mask first.
    """
    print("\n--- Starting Balanced Patch Generation ---")
    print(f"Patch Size: {patch_size}x{patch_size}, Stride: {stride}")
    print(f"Original Images: {image_dir}")
    print(f"Original Masks Base: {mask_base_dir}")
    print(f"Target Class: {target_class_name}, Subdir: {target_subdir_name}")
    print(f"Saving Balanced Patches (Images) to: {patch_save_img_dir}")
    print(f"Saving Balanced Patches (Masks Base) to: {patch_save_mask_dir_base}")

    patch_save_mask_dir_specific = os.path.join(patch_save_mask_dir_base, target_subdir_name)
    os.makedirs(patch_save_img_dir, exist_ok=True)
    os.makedirs(patch_save_mask_dir_specific, exist_ok=True)

    # Clean existing patch directories (optional, but recommended)
    print("Cleaning previous balanced patch directories (if they exist)...")
    if os.path.exists(patch_save_img_dir): shutil.rmtree(patch_save_img_dir); os.makedirs(patch_save_img_dir)
    if os.path.exists(patch_save_mask_dir_specific): shutil.rmtree(patch_save_mask_dir_specific); os.makedirs(patch_save_mask_dir_specific)

    # Find original images
    original_image_paths = []
    for ext in ["*.jpg", "*.png", "*.tif", "*.jpeg", "*.bmp"]:
        original_image_paths.extend(sorted(glob.glob(os.path.join(image_dir, ext))))
        original_image_paths.extend(sorted(glob.glob(os.path.join(image_dir, ext.upper()))))
    # <<< MODIFICATION: Filter based on presence of mask only >>>
    # Only process images that HAVE a mask for the target lesion type
    valid_original_paths = []
    print("Checking for corresponding masks...")
    for img_path in tqdm(original_image_paths, desc="Checking Masks", leave=False):
         img_id_ext = os.path.basename(img_path)
         img_id, _ = os.path.splitext(img_id_ext)
         original_mask_filename = f"{img_id}_{target_class_name}.tif"
         original_mask_path = os.path.join(mask_base_dir, target_subdir_name, original_mask_filename)
         if os.path.exists(original_mask_path):
             valid_original_paths.append(img_path)
         # else: # Optional: print which images are skipped
         #     print(f"Skipping {img_id_ext}, no mask found at {original_mask_path}")

    original_image_paths = sorted(list(set(p for p in valid_original_paths if "_patch_" not in os.path.basename(p)))) # Avoid re-patching


    print(f"Found {len(original_image_paths)} original images with corresponding masks to extract patches from.")
    lesion_patches = [] # Stores tuples: (img_patch_path, mask_patch_path, original_img_id)
    non_lesion_patches = []
    skipped_missing_mask = 0 # Should be 0 now due to pre-filtering
    error_count = 0

    for img_path in tqdm(original_image_paths, desc="Extracting Patches"):
        img_id_ext = os.path.basename(img_path)
        img_id, img_ext = os.path.splitext(img_id_ext)

        original_mask_filename = f"{img_id}_{target_class_name}.tif"
        original_mask_path = os.path.join(mask_base_dir, target_subdir_name, original_mask_filename)

        # This check should technically not be needed anymore due to pre-filtering, but keep as safeguard
        if not os.path.exists(original_mask_path):
            skipped_missing_mask += 1
            continue

        try:
            image = cv2.imread(img_path) # BGR
            mask = cv2.imread(original_mask_path, cv2.IMREAD_GRAYSCALE)
            if image is None: raise IOError(f"Failed to read image: {img_path}")
            if mask is None: raise IOError(f"Failed to read mask: {original_mask_path}")

            # --- Apply FoV Mask First ---
            fov_mask = create_fov_mask(image, threshold=15)
            if fov_mask is None: raise ValueError("FoV mask creation failed.")
            fov_mask_3ch = cv2.cvtColor(fov_mask, cv2.COLOR_GRAY2BGR)
            image_fov = cv2.bitwise_and(image, fov_mask_3ch) # Apply to BGR image
            mask_binary = (mask > 0).astype(np.uint8) # 0/1
            mask_fov = cv2.bitwise_and(mask_binary, fov_mask) # Apply to binary mask

            # Get image dimensions
            h, w = image_fov.shape[:2]

            # --- Extract Overlapping Patches ---
            patch_count_this_img = 0
            for y in range(0, h - patch_size + 1, stride):
                for x in range(0, w - patch_size + 1, stride):
                    img_patch = image_fov[y:y+patch_size, x:x+patch_size]
                    mask_patch = mask_fov[y:y+patch_size, x:x+patch_size] # Binary (0/1)

                    # Check if patch is mostly black (outside FoV) - skip if >95% black
                    if np.mean(img_patch) < (0.05 * 255): continue

                    # Define save paths for this patch
                    patch_img_filename = f"{img_id}_patch_{y}_{x}{img_ext}"
                    patch_mask_filename = f"{img_id}_{target_class_name}_patch_{y}_{x}.tif"
                    patch_img_save_path = os.path.join(patch_save_img_dir, patch_img_filename)
                    patch_mask_save_path = os.path.join(patch_save_mask_dir_specific, patch_mask_filename)

                    # Check if the mask patch contains any lesion pixels
                    if np.sum(mask_patch) > 0: # Lesion patch
                        lesion_patches.append((img_patch, mask_patch, patch_img_save_path, patch_mask_save_path))
                    else: # Non-lesion patch
                        non_lesion_patches.append((img_patch, mask_patch, patch_img_save_path, patch_mask_save_path))
                    patch_count_this_img += 1
            # Optional: print how many patches came from this image
            # print(f"  - Extracted {patch_count_this_img} valid patches from {img_id_ext}")


        except Exception as e:
            print(f"ERROR processing {img_id}: {e}")
            error_count += 1
            # import traceback; traceback.print_exc() # Uncomment for detailed error

    # --- Balancing ---
    num_lesion_patches = len(lesion_patches)
    num_non_lesion_patches = len(non_lesion_patches)
    print(f"\nExtracted {num_lesion_patches} lesion patches and {num_non_lesion_patches} non-lesion patches.")

    if num_lesion_patches == 0:
        print("WARNING: No lesion patches found! Cannot balance. Saving only non-lesion patches (if any).")
        selected_patches = non_lesion_patches # Save all if no positive examples
    elif num_non_lesion_patches == 0:
         print("WARNING: No non-lesion patches found! Cannot balance. Saving only lesion patches.")
         selected_patches = lesion_patches
    elif num_non_lesion_patches < num_lesion_patches:
        print(f"Balancing: Fewer non-lesion ({num_non_lesion_patches}) than lesion patches ({num_lesion_patches}). Keeping all non-lesion and sampling {num_non_lesion_patches} lesion patches.")
        selected_lesion = random.sample(lesion_patches, num_non_lesion_patches) # Sample down positives
        selected_patches = selected_lesion + non_lesion_patches # Combine
        random.shuffle(selected_patches) # Shuffle the final list
    else: # More non-lesion than lesion (typical case)
        print(f"Balancing: Selecting {num_lesion_patches} non-lesion patches to match {num_lesion_patches} lesion patches.")
        selected_non_lesion = random.sample(non_lesion_patches, num_lesion_patches)
        selected_patches = lesion_patches + selected_non_lesion
        random.shuffle(selected_patches) # Shuffle the final list

    print(f"Total patches after balancing: {len(selected_patches)}")

    # --- Save Selected Balanced Patches ---
    saved_patch_count = 0
    for img_patch, mask_patch, img_save_path, mask_save_path in tqdm(selected_patches, desc="Saving Balanced Patches"):
        try:
            cv2.imwrite(img_save_path, img_patch) # Save image patch (BGR)
            cv2.imwrite(mask_save_path, mask_patch * 255) # Save mask patch (0/255)
            saved_patch_count += 1
        except Exception as e_save:
            print(f"ERROR saving patch pair ({os.path.basename(img_save_path)}, {os.path.basename(mask_save_path)}): {e_save}")
            error_count += 1

    print(f"\n--- Balanced Patch Generation Finished ---")
    print(f"Saved {saved_patch_count} balanced image/mask patch pairs.")
    # if skipped_missing_mask > 0: print(f"Skipped {skipped_missing_mask} original images due to missing masks.") # Should be 0 now
    if error_count > 0: print(f"Encountered {error_count} errors during processing/saving.")
    return saved_patch_count


# --- REMOVED/COMMENTED OUT Offline Augmentation Function ---
# def augment_patches(...):
#     """ This function is no longer used in the main workflow. """
#     print("\n--- SKIPPING Offline Geometric Augmentation of Balanced Patches ---")
#     # Return dummy values or raise error if called accidentally
#     return 0, 0
#     # ... (rest of the function definition commented out or deleted)


# --- MODIFIED Online Training Transforms (Added CLAHE) ---
def get_train_transforms(img_size=IMAGE_SIZE): # img_size should match PATCH_SIZE
    print(f"--- Using online transforms for patches (Size: {img_size}x{img_size}) with ONLINE CLAHE ---")
    # Apply CLAHE first before other intensity/color augmentations
    return A.Compose([
        A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), always_apply=True, p=1.0), # Apply CLAHE online
        # --- Keep other online augmentations ---
        A.ElasticTransform(p=0.3, alpha=15, sigma=img_size*0.04, alpha_affine=img_size*0.03,
                           border_mode=cv2.BORDER_CONSTANT, value=0, mask_value=0),
        A.GridDistortion(p=0.3, border_mode=cv2.BORDER_CONSTANT, value=0, mask_value=0, distort_limit=0.4),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.7),
        A.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.05, p=0.5),
        A.GaussNoise(p=0.3, var_limit=(10.0, 50.0)),
        A.GaussianBlur(blur_limit=(3, 7), p=0.3),
        A.MedianBlur(blur_limit=5, p=0.2),
        A.CoarseDropout(max_holes=8, max_height=img_size//10, max_width=img_size//10,
                        min_holes=1, min_height=img_size//20, min_width=img_size//20,
                        fill_value=0, mask_fill_value=0, p=0.3),
        A.Normalize(mean=_NORM_MEAN, std=_NORM_STD),
        ToTensorV2(),
    ])

# MODIFIED Validation Transforms (Added CLAHE)
def get_val_transforms(img_size=IMAGE_SIZE): # img_size should match PATCH_SIZE
    print(f"--- Using validation transforms for patches (Size: {img_size}x{img_size}) with ONLINE CLAHE ---")
    # Apply CLAHE consistently in validation before normalization
    return A.Compose([
        A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), always_apply=True, p=1.0), # Apply CLAHE online
        A.Normalize(mean=_NORM_MEAN, std=_NORM_STD),
        ToTensorV2(),
    ])

print("Preprocessing and Augmentation functions defined correctly (Patching only + Online CLAHE).")

# Block 4: Dataset and Data Entry Creation (CLEANED Dataset - Assuming 384x384 Patches)

# --- Dataset Class (REMOVED Interpolation Logic) ---
class IdridDataset(Dataset):
    def __init__(self, image_paths, mask_dict_list, class_names, transform=None, image_size=IMAGE_SIZE):
        self.image_paths = image_paths; self.mask_dict_list = mask_dict_list
        self.transform = transform; self.class_names = class_names
        self.num_classes = len(class_names); self.image_size = image_size # Should be 384
        if len(image_paths) != len(mask_dict_list): raise ValueError("Img/Mask mismatch")
        print(f"Initialized Dataset with {len(self.image_paths)} patch samples for class '{self.class_names[0]}'.")
        print(f"Expecting patches of size {self.image_size}x{self.image_size}.") # Changed print

    def __len__(self): return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]; mask_paths_dict = self.mask_dict_list[idx]
        image = None; h_orig, w_orig = -1, -1
        target_class_name = self.class_names[0]

        try:
            image = cv2.imread(img_path);
            if image is None: raise IOError(f"cv2.imread returned None for {img_path}")
            # <<< Check loaded patch size matches expected IMAGE_SIZE >>>
            h_orig, w_orig = image.shape[:2]
            if h_orig != self.image_size or w_orig != self.image_size:
                 print(f"WARN: Loaded image patch {img_path} has size {(h_orig, w_orig)}, expected {(self.image_size, self.image_size)}. Resizing.")
                 image = cv2.resize(image, (self.image_size, self.image_size), interpolation=cv2.INTER_LINEAR)
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB);
        except Exception as e:
            print(f"ERR load img PATCH {img_path}: {e}"); return torch.zeros((3, self.image_size, self.image_size)), torch.zeros((self.num_classes, self.image_size, self.image_size))

        masks_np = []; mask_path = mask_paths_dict.get(target_class_name); current_mask_np = None

        if mask_path and os.path.exists(mask_path):
            try:
                current_mask_np = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
                if current_mask_np is None: raise IOError(f"cv2.imread returned None for {mask_path}")
                 # <<< Check loaded mask patch size matches expected IMAGE_SIZE >>>
                mh, mw = current_mask_np.shape[:2]
                if mh != self.image_size or mw != self.image_size:
                    print(f"WARN: Loaded mask patch {mask_path} has size {(mh, mw)}, expected {(self.image_size, self.image_size)}. Resizing.")
                    current_mask_np = cv2.resize(current_mask_np, (self.image_size, self.image_size), interpolation=cv2.INTER_NEAREST)
                current_mask_np = (current_mask_np > 0).astype(np.uint8) # Binarize
            except Exception as e_mask:
                print(f"ERR load mask PATCH {mask_path}: {e_mask}"); current_mask_np = None
        else:
             if mask_path: print(f"WARN: Mask patch path specified but not found: {mask_path}")

        if current_mask_np is None:
            print(f"WARN: Creating zero mask for image patch {img_path} as mask was missing or failed to load.")
            # Use target size directly
            current_mask_np = np.zeros((self.image_size, self.image_size), dtype=np.uint8)

        masks_np.append(current_mask_np)

        if self.transform:
            try:
                augmented = self.transform(image=image, masks=masks_np)
                image_tensor = augmented['image']
                masks_list = augmented['masks']
                masks_tensor = torch.stack(masks_list, dim=0).float()
            except Exception as e:
                print(f"ERR transform PATCH {img_path}: {e}"); return torch.zeros((3, self.image_size, self.image_size)), torch.zeros((self.num_classes, self.image_size, self.image_size))
        else:
            print("Warning: No transforms applied! Using basic to_tensor.");
            image_tensor = torch.from_numpy(image).permute(2,0,1).float()/255.0
            masks_tensor=torch.stack([torch.from_numpy(m) for m in masks_np],dim=0).float()

        # Final checks... should match now if transforms don't change size unexpectedly
        final_h, final_w = self.image_size, self.image_size
        expected_img_shape=(3, final_h, final_w)
        expected_mask_shape=(self.num_classes, final_h, final_w)

        # <<< REMOVED the F.interpolate calls here - rely on transforms / initial load check >>>
        # if image_tensor.shape[1:] != (final_h, final_w): ...
        # if masks_tensor.shape[1:] != (final_h, final_w): ...

        if image_tensor.shape != expected_img_shape:
            print(f"ERR img shape {idx} {image_tensor.shape} != {expected_img_shape}"); return torch.zeros(expected_img_shape), torch.zeros(expected_mask_shape)
        if masks_tensor.shape != expected_mask_shape:
             print(f"ERR mask shape {idx} {masks_tensor.shape} != {expected_mask_shape}"); return torch.zeros(expected_img_shape), torch.zeros(expected_mask_shape)

        return image_tensor, masks_tensor


# --- create_data_entries Function (Unchanged from previous version - already simplified) ---
def create_data_entries(image_dir, mask_base_dir): # Reads from BALANCED PATCH dir
    target_class_name = CLASS_NAMES[0] # e.g., 'EX'
    target_subdir_name = MASK_SUBDIR_NAMES[target_class_name] # e.g., '3. Hard Exudates'

    print(f"\n--- Creating Data Entries ({target_class_name} - From BALANCED Patches) ---") # Changed title
    print(f"Searching Image Patch Dir: {image_dir}")
    print(f"Mask Patch Base Dir: {mask_base_dir}")
    print(f"Target Class: '{target_class_name}', Subdir: '{target_subdir_name}'")

    if not os.path.isdir(image_dir): print(f"ERROR: Image patch dir not found: {image_dir}"); return []
    if not os.path.isdir(mask_base_dir): print(f"ERROR: Mask patch base dir not found: {mask_base_dir}"); return []

    entries = []; missing_masks = 0
    image_patch_paths = []
    # Find ALL image patches (ONLY balanced originals now)
    print(f"Searching for all image patches in {image_dir}")
    for ext in ["*.jpg", "*.png", "*.tif", "*.jpeg", "*.bmp"]:
        image_patch_paths.extend(sorted(glob.glob(os.path.join(image_dir, ext))))
        image_patch_paths.extend(sorted(glob.glob(os.path.join(image_dir, ext.upper()))))
    # <<< SIMPLIFIED FILTER: Only expect balanced patch format >>>
    image_patch_paths = sorted(list(set(p for p in image_patch_paths if "_patch_" in os.path.basename(p) and "_aug_" not in os.path.basename(p) )))

    print(f"Found {len(image_patch_paths)} potential balanced image patch files.")
    if not image_patch_paths: print(f"Warning: No image patches found in {image_dir}. Check generate_balanced_patches output."); return []

    mask_search_dir = os.path.join(mask_base_dir, target_subdir_name)
    print(f"Pairing image patches with '{target_class_name}' mask patches from: {mask_search_dir}")
    if not os.path.isdir(mask_search_dir): print(f"ERROR: Mask patch search directory not found: {mask_search_dir}"); return []

    for img_patch_path in tqdm(image_patch_paths, desc=f"Pairing {target_class_name} mask patches"):
        img_patch_filename = os.path.basename(img_patch_path)
        img_id_patch_base, img_ext = os.path.splitext(img_patch_filename) # e.g., IDRiD_01_patch_y_x

        # --- SIMPLIFIED Logic to Construct Corresponding Mask Patch Filename ---
        mask_patch_filename = None
        parts = img_id_patch_base.split('_patch_')
        if len(parts) == 2:
            base_img_id = parts[0]; coord_part = parts[1]
            mask_patch_filename = f"{base_img_id}_{target_class_name}_patch_{coord_part}.tif"
        else: print(f"DEBUG: Could not parse patch filename format: {img_patch_filename}")

        potential_mask_path = os.path.join(mask_search_dir, mask_patch_filename) if mask_patch_filename else None
        masks_dict = {}
        if potential_mask_path and os.path.exists(potential_mask_path):
            masks_dict[target_class_name] = potential_mask_path
        else:
            if potential_mask_path: print(f"DEBUG: Mask patch pairing issue. Img: {img_patch_filename}, Expected Mask: {mask_patch_filename} (Not found in {mask_search_dir})")
            else: print(f"DEBUG: Could not construct expected mask patch filename for {img_patch_filename}")
            masks_dict[target_class_name] = None; missing_masks += 1

        if os.path.exists(img_patch_path):
            if masks_dict.get(target_class_name): entries.append({'image_path': img_patch_path, 'mask_paths': masks_dict})
            else: print(f"Skipping entry for {img_patch_filename} due to missing/unpaired mask.")

    print("\n--- Mask Patch File Check Report ---")
    if missing_masks > 0:
        print(f"  Found {len(entries)} valid image-mask patch pairs.")
        print(f"  {missing_masks} potential issues encountered (missing mask file or filename mismatch). Check DEBUG messages.")
    else: print(f"  All {len(entries)} expected '{target_class_name}' mask patch files were found and paired.")
    print("------------------------------")
    print(f"\nTotal valid data entries created from balanced patches: {len(entries)}")
    if not entries: print(f"WARNING: Could not create any valid data entries from balanced patches.")
    print(f"--- End Creating Data Entries ---\n")
    return entries

# Block 5: Model Definition (Unchanged - Assuming IMAGE_SIZE was updated in config)
# --- Attention Gate Module (Unchanged) ---
class AttentionGate(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g = nn.Sequential(nn.Conv2d(F_g, F_int, 1, 1, 0, bias=True), nn.BatchNorm2d(F_int))
        self.W_x = nn.Sequential(nn.Conv2d(F_l, F_int, 1, 1, 0, bias=True), nn.BatchNorm2d(F_int))
        self.psi = nn.Sequential(nn.Conv2d(F_int, 1, 1, 1, 0, bias=True), nn.BatchNorm2d(1), nn.Sigmoid())
        self.relu = nn.ReLU(inplace=True)
    def forward(self, g, x):
        g1 = self.W_g(g); x1 = self.W_x(x)
        if g1.shape[2:] != x1.shape[2:]: x1 = F.interpolate(x1, size=g1.shape[2:], mode='bilinear', align_corners=False)
        psi = self.relu(g1 + x1); psi = self.psi(psi)
        return x * psi

# --- DecoderBlock with Attention Gate (Unchanged) ---
class DecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels, use_attention=True):
        super().__init__(); self.use_attention = use_attention
        up_out_channels = in_channels // 2; combined_channels = up_out_channels + skip_channels
        self.up = nn.ConvTranspose2d(in_channels, up_out_channels, kernel_size=2, stride=2)
        if self.use_attention:
            print(f"  - DecoderBlock: Using Attention Gate (F_g={up_out_channels}, F_l={skip_channels}, F_int={skip_channels//2})")
            self.attention_gate = AttentionGate(F_g=up_out_channels, F_l=skip_channels, F_int=skip_channels//2)
            conv_in_channels = combined_channels
        else:
            print(f"  - DecoderBlock: Not using Attention Gate.")
            self.attention_gate = None; conv_in_channels = combined_channels
        self.conv_block = nn.Sequential( nn.Conv2d(conv_in_channels, out_channels, 3, padding=1, bias=False), nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True), nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False), nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True) )
    def forward(self, x, skip=None):
        x_up = self.up(x)
        if skip is not None:
            if x_up.shape[2:] != skip.shape[2:]: skip_aligned = F.interpolate(skip, size=x_up.shape[2:], mode='bilinear', align_corners=False)
            else: skip_aligned = skip
            if self.use_attention and self.attention_gate is not None: skip_final = self.attention_gate(g=x_up, x=skip_aligned)
            else: skip_final = skip_aligned
            x = torch.cat([x_up, skip_final], dim=1)
        else: x = x_up
        x = self.conv_block(x); return x

# --- CustomSwinUNet (Unchanged - Reads IMAGE_SIZE from config) ---
class CustomSwinUNet(nn.Module):
    def __init__(self, encoder=ENCODER, encoder_weights=ENCODER_WEIGHTS, classes=NUM_CLASSES,
                 decoder_channels=(256, 128, 64, 32), use_attention_gates=True,
                 debug_prints=DEBUG_FORWARD_PASS):
        super().__init__()
        self.classes = classes; self.debug_prints = debug_prints; self.use_attention_gates = use_attention_gates
        if len(decoder_channels) != 4: raise ValueError("decoder_channels must have 4 values.")
        self.encoder = timm.create_model( encoder, pretrained=(encoder_weights == 'imagenet'), features_only=True, img_size=IMAGE_SIZE, pretrained_window_sizes=(0, 0, 0, 0) ) # Uses IMAGE_SIZE
        encoder_out_channels = self.encoder.feature_info.channels()
        print(f"Using encoder: {encoder} (Requires input size {IMAGE_SIZE})")
        print(f"Encoder feature channels: {encoder_out_channels}")
        if len(encoder_out_channels) < 4: print(f"Warning: Encoder levels < 4")
        enc_ch_reversed = encoder_out_channels[::-1]
        print(f"Attention Gates Enabled: {self.use_attention_gates}")
        print(f"Decoder Block 1 Channels: In={enc_ch_reversed[0]}, Skip={enc_ch_reversed[1]}, Out={decoder_channels[0]}")
        self.decoder1 = DecoderBlock(enc_ch_reversed[0], enc_ch_reversed[1], decoder_channels[0], use_attention=self.use_attention_gates)
        print(f"Decoder Block 2 Channels: In={decoder_channels[0]}, Skip={enc_ch_reversed[2]}, Out={decoder_channels[1]}")
        self.decoder2 = DecoderBlock(decoder_channels[0], enc_ch_reversed[2], decoder_channels[1], use_attention=self.use_attention_gates)
        print(f"Decoder Block 3 Channels: In={decoder_channels[1]}, Skip={enc_ch_reversed[3]}, Out={decoder_channels[2]}")
        self.decoder3 = DecoderBlock(decoder_channels[1], enc_ch_reversed[3], decoder_channels[2], use_attention=self.use_attention_gates)
        self.decoder4 = nn.ConvTranspose2d(decoder_channels[2], decoder_channels[3], kernel_size=2, stride=2)
        self.final_relu = nn.ReLU(inplace=True)
        self.segmentation_head = nn.Conv2d(decoder_channels[3], self.classes, kernel_size=1)
    def forward(self, x):
        input_shape = x.shape; features = self.encoder(x); last_feature = features[-1]
        if last_feature.ndim == 4 and self.encoder.feature_info and self.encoder.feature_info.channels()[-1]!=last_feature.shape[1] and self.encoder.feature_info.channels()[-1]==last_feature.shape[-1]:
            features = [f.permute(0, 3, 1, 2).contiguous() for f in features]
        if len(features) < 4: raise ValueError(f"Encoder needs >= 4 features for this decoder")
        skips = features[:len(features)-1][::-1]; decoder_input = features[-1]
        d1 = self.decoder1(decoder_input, skips[0]); d2 = self.decoder2(d1, skips[1]); d3 = self.decoder3(d2, skips[2])
        d4 = self.decoder4(d3); d4 = self.final_relu(d4); logits = self.segmentation_head(d4)
        if logits.shape[2:] != input_shape[2:]: logits = F.interpolate(logits, size=input_shape[2:], mode='bilinear', align_corners=False)
        return logits

# --- build_model (Unchanged) ---
def build_model(encoder=ENCODER, encoder_weights=ENCODER_WEIGHTS, classes=NUM_CLASSES, activation=None, use_attention=True):
    print(f"Building CustomSwinUNet: Encoder='{encoder}', Weights='{encoder_weights}', Classes={classes}, Attention={use_attention}")
    model = CustomSwinUNet( encoder=encoder, encoder_weights=encoder_weights, classes=classes, use_attention_gates=use_attention, debug_prints=DEBUG_FORWARD_PASS )
    if activation is not None: print(f"Warning: Activation '{activation}' ignored in build_model.")
    return model


# Block 6: Loss & Metrics (Unchanged)
# --- BCEDiceLoss Class (Unchanged) ---
class BCEDiceLoss(nn.Module):
    def __init__(self, weight_bce=0.5, weight_dice=0.5, dice_smooth=1.0):
        super().__init__(); self.weight_bce = weight_bce; self.weight_dice = weight_dice
        self.bce_loss = nn.BCEWithLogitsLoss()
        self.dice_loss = DiceLoss(mode='binary', from_logits=True, smooth=dice_smooth)
        print(f"Initializing BCEDiceLoss: weight_bce={weight_bce}, weight_dice={weight_dice}")
    def forward(self, y_pred_logits, y_true):
        if y_true.dtype != torch.float32: y_true = y_true.float()
        if y_pred_logits.shape != y_true.shape: raise ValueError(f"Shape mismatch: Pred {y_pred_logits.shape}, Target {y_true.shape}")
        bce = self.bce_loss(y_pred_logits, y_true); dice = self.dice_loss(y_pred_logits, y_true)
        loss = (self.weight_bce * bce) + (self.weight_dice * dice)
        return loss

# --- calculate_metrics Function (Unchanged - uses CLASS_NAMES) ---
def calculate_metrics(y_true, y_pred_logits, threshold=0.5):
    target_class_name = CLASS_NAMES[0] # 'EX'
    batch_size=y_true.shape[0]; iou_batch_total=0.0; dice_batch_total=0.0
    iou_key = f"iou ({target_class_name})"; dice_key = f"dice ({target_class_name})"
    if batch_size==0: return {iou_key:0.0, dice_key:0.0}
    y_pred_binary=(torch.sigmoid(y_pred_logits)>threshold).byte(); y_true_byte=y_true.byte()
    for i in range(batch_size):
        pred_cls=y_pred_binary[i,0]; true_cls=y_true_byte[i,0]
        pred_flat=pred_cls.reshape(-1); true_flat=true_cls.reshape(-1)
        intersection=torch.sum(pred_flat & true_flat).item(); union=torch.sum(pred_flat | true_flat).item()
        tp=intersection; fp=torch.sum(pred_flat & (~true_flat)).item(); fn=torch.sum((~pred_flat) & true_flat).item()
        iou_lesion=float(intersection)/union if union>0 else (1.0 if intersection==0 and union==0 else 0.0)
        dice_denom=(2*tp)+fp+fn; dice_lesion=(2.0*tp)/dice_denom if dice_denom > 0 else (1.0 if (tp+fp+fn == 0) else 0.0)
        iou_batch_total+=iou_lesion; dice_batch_total+=dice_lesion
    mean_iou=iou_batch_total/batch_size; mean_dice=dice_batch_total/batch_size
    return {iou_key: mean_iou, dice_key: mean_dice}


# Block 7: Training & Validation Loops (Unchanged Internally)
# --- train_one_epoch (Unchanged) ---
def train_one_epoch(model, loader, optimizer, criterion, device, scheduler=None, is_batch_scheduler=False):
    model.train(); epoch_loss = 0.0
    pbar = tqdm(enumerate(loader), total=len(loader), desc="Train")
    for i, (images, masks) in pbar:
        images = images.to(device, dtype=torch.float32); masks = masks.to(device, dtype=torch.float32)
        optimizer.zero_grad(); outputs = model(images); loss = criterion(outputs, masks)
        if torch.isnan(loss): print(f"WARNING: NaN loss detected at batch {i}. Skipping backward."); continue
        loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0); optimizer.step()
        if scheduler and is_batch_scheduler: scheduler.step()
        epoch_loss += loss.item(); pbar.set_postfix(loss=loss.item())
    avg_loss = epoch_loss / len(loader) if len(loader) > 0 else 0.0; pbar.close()
    return avg_loss, {}

# --- validate_one_epoch (Unchanged - uses dynamic keys from calculate_metrics) ---
def validate_one_epoch(model, loader, criterion, device, epoch, num_epochs, visualize=False, vis_dir=VIS_DIR, num_vis_samples=1):
    model.eval(); epoch_loss = 0.0
    target_class_name = CLASS_NAMES[0]; iou_key = f"iou ({target_class_name})"; dice_key = f"dice ({target_class_name})"
    metric_keys = [iou_key, dice_key]; all_metrics = {k: [] for k in metric_keys}
    visualized_count = 0
    pbar = tqdm(enumerate(loader), total=len(loader), desc="Val")
    with torch.no_grad():
        for i, (images, masks) in pbar:
            images = images.to(device, dtype=torch.float32); masks = masks.to(device, dtype=torch.float32)
            outputs = model(images); loss = criterion(outputs, masks)
            if not torch.isnan(loss): epoch_loss += loss.item()
            else: print(f"WARNING: NaN loss detected during validation batch {i}.")
            batch_metrics = calculate_metrics(masks, outputs)
            for key in metric_keys: all_metrics[key].append(batch_metrics.get(key, 0.0))
            pbar.set_postfix(loss=loss.item() if not torch.isnan(loss) else float('nan'), iou=batch_metrics.get(iou_key, 0.0))
            # --- Visualization ---
            if visualize and visualized_count < num_vis_samples and images.shape[0] > 0:
                try:
                    img_tensor=images[0]; mask_tensor=masks[0]; output_logits=outputs[0]
                    label_str=CLASS_NAMES[0]; color_str=VIS_COLORS[0] # Get current class info
                    img_vis=img_tensor.cpu().numpy().transpose(1,2,0); img_vis=img_vis*np.array(_NORM_STD)+np.array(_NORM_MEAN); img_vis=np.clip(img_vis,0,1); img_vis=(img_vis*255).astype(np.uint8)
                    pred_probs=torch.sigmoid(output_logits[0]).cpu().numpy(); pred_binary=(pred_probs > 0.5).astype(np.uint8)
                    mask_binary=(mask_tensor[0].cpu().numpy() > 0).astype(np.uint8)
                    h,w=img_vis.shape[:2]; gt_overlay=np.zeros((h,w,3),dtype=np.uint8); pred_overlay=np.zeros((h,w,3),dtype=np.uint8)
                    color_rgb=np.array(mcolors.to_rgb(color_str))*255
                    gt_overlay[mask_binary==1]=color_rgb; pred_overlay[pred_binary==1]=color_rgb
                    alpha=0.6; gt_display=cv2.addWeighted(gt_overlay,alpha,img_vis,1-alpha,0); pred_display=cv2.addWeighted(pred_overlay,alpha,img_vis,1-alpha,0)
                    fig,ax=plt.subplots(1,4,figsize=(24,6))
                    ax[0].imshow(img_vis);ax[0].set_title("Input");ax[0].axis('off'); ax[1].imshow(gt_display);ax[1].set_title(f"Ground Truth ({label_str})");ax[1].axis('off'); ax[2].imshow(pred_display);ax[2].set_title(f"Prediction ({label_str}) @ 0.5");ax[2].axis('off');
                    im=ax[3].imshow(pred_probs,cmap='viridis',vmin=0,vmax=1); ax[3].set_title(f"Prediction Probabilities ({label_str})"); ax[3].axis('off'); fig.colorbar(im,ax=ax[3],fraction=0.046,pad=0.04)
                    patches_list=[patches.Patch(color=color_str, label=label_str)]; fig.legend(handles=patches_list,loc='lower center',ncol=1,bbox_to_anchor=(0.5,-0.02))
                    plt.suptitle(f"Epoch {epoch+1}/{num_epochs} - Val Sample {(i*loader.batch_size + 0):03d}",fontsize=14); plt.tight_layout(rect=[0,0.05,1,0.97])
                    vis_filename=f"epoch_{(epoch+1):03d}_sample_{(i*loader.batch_size):03d}_validation.png"; vis_path=os.path.join(vis_dir,vis_filename)
                    plt.savefig(vis_path,bbox_inches='tight',dpi=150); plt.close(fig); visualized_count+=1
                except Exception as e_vis:
                    print(f"WARN: Error during visualization: {e_vis}")
                    if 'fig' in locals() and plt.fignum_exists(fig.number): plt.close(fig)
    # Calculate average loss (simple average, assumes few NaNs if any)
    avg_loss=epoch_loss/len(loader) if len(loader)>0 else 0.0
    avg_metrics={key: np.mean(values) if values else 0.0 for key, values in all_metrics.items()}
    pbar.close()
    return avg_loss, avg_metrics


# Block 8: Utilities (Unchanged - uses dynamic keys)
# --- save_checkpoint (Unchanged) ---
def save_checkpoint(state, filename):
    print(f" => Saving checkpoint to {filename}"); torch.save(state, filename)

# --- load_checkpoint (Unchanged - handles dynamic keys) ---
def load_checkpoint(model, optimizer, scheduler, filename):
    target_class_name=CLASS_NAMES[0]; iou_key=f"iou ({target_class_name})"; dice_key=f"dice ({target_class_name})"
    history_keys=['train_loss', 'val_loss', f'val_{iou_key}', f'val_{dice_key}']
    history={k:[] for k in history_keys}; start_epoch=0; best_metric=float('-inf')
    if filename and os.path.isfile(filename):
        print(f" => Loading checkpoint '{filename}'"); checkpoint=torch.load(filename, map_location=DEVICE,weights_only=False)
        strict_loading = True; ckpt_class_names = checkpoint.get('class_names', None)
        if ckpt_class_names and ckpt_class_names != CLASS_NAMES: print(f"WARN: Checkpoint class {ckpt_class_names} != current {CLASS_NAMES}. Forcing non-strict."); strict_loading=False
        if 'model_state_dict' in checkpoint:
            print(f"Attempting to load model state (strict={strict_loading})...")
            try: model.load_state_dict(checkpoint['model_state_dict'], strict=strict_loading); print("Model state loaded.")
            except Exception as e: print(f"ERR load model state dict: {e}.")
        else: print("Warn: 'model_state_dict' not found.")
        if optimizer and 'optimizer_state_dict' in checkpoint and strict_loading:
            try: optimizer.load_state_dict(checkpoint['optimizer_state_dict']); print("Optimizer state loaded.")
            except Exception as e: print(f"Warn load optimizer: {e}.")
        elif optimizer: print("Optimizer state not loaded.")
        if scheduler and 'scheduler_state_dict' in checkpoint and hasattr(scheduler, 'load_state_dict') and strict_loading:
            try: scheduler.load_state_dict(checkpoint['scheduler_state_dict']); print("Scheduler state loaded.")
            except Exception as e: print(f"Warn load scheduler: {e}.")
        elif scheduler: print("Scheduler state not loaded.")
        if not strict_loading: print("Non-strict load. Resetting epoch, best metric, history."); start_epoch=0; best_metric=float('-inf'); history={k:[] for k in history_keys}
        else:
            start_epoch=checkpoint.get('epoch', -1)+1; best_metric=checkpoint.get('best_metric', float('-inf')); loaded_history=checkpoint.get('history', {})
            for key in history_keys: history[key]=loaded_history.get(key, [])
            if not any(history.values()): print("WARN: Loaded history empty.")
            print(f" => Resuming from epoch {start_epoch}. Best Val IoU ({target_class_name}): {best_metric:.4f}")
    else:
        if filename: print(f" => No checkpoint found @ '{filename}'. Starting fresh.")
        else: print(" => No checkpoint filename provided. Starting fresh.")
        start_epoch=0; best_metric=float('-inf'); history={k:[] for k in history_keys}
    for key in history_keys: history.setdefault(key, [])
    return start_epoch, best_metric, history

# --- plot_training_curves (Unchanged - uses dynamic keys) ---
def plot_training_curves(history, save_path=TRAINING_PLOTS_PATH):
    target_class_name=CLASS_NAMES[0]; iou_key=f"val_iou ({target_class_name})"; dice_key=f"val_dice ({target_class_name})"
    epochs=range(1, len(history.get('train_loss', []))+1)
    if not epochs: print("History empty, cannot plot."); return
    plt.figure(figsize=(18,6));
    plt.subplot(1,3,1); plt.plot(epochs, history.get('train_loss',[]),'bo-',label='Train Loss'); plt.plot(epochs, history.get('val_loss',[]),'ro-',label='Val Loss'); plt.title('Loss'); plt.xlabel('Epochs');plt.ylabel('Loss');plt.legend();plt.grid(True)
    plt.subplot(1,3,2); iou_data=history.get(iou_key,[]);
    if iou_data: plt.plot(epochs, iou_data,'go-',label=f'Val IoU ({target_class_name})');
    plt.title(f'Val IoU ({target_class_name})'); plt.xlabel('Epochs');plt.ylabel('IoU');plt.legend();plt.grid(True);plt.ylim(0,1)
    plt.subplot(1,3,3); dice_data=history.get(dice_key,[]);
    if dice_data: plt.plot(epochs, dice_data,'mo-',label=f'Val Dice ({target_class_name})');
    plt.title(f'Val Dice ({target_class_name})'); plt.xlabel('Epochs');plt.ylabel('Dice');plt.legend();plt.grid(True);plt.ylim(0,1)
    plt.suptitle(f"Training Curves: {MODEL_NAME}",fontsize=14); plt.tight_layout(rect=[0,0.03,1,0.95])
    try: plt.savefig(save_path); print(f"Training curves saved: {save_path}")
    except Exception as e: print(f"ERR save curves: {e}")
    plt.close()


# Block 9: Main Execution Block (MODIFIED Workflow for Patching + Skip Generation)

if __name__ == '__main__':
    # <<< --- Get settings from Block 2 --- >>>
    target_class_name = CLASS_NAMES[0] # 'EX'
    target_subdir_name = MASK_SUBDIR_NAMES[target_class_name] # e.g., '3. Hard Exudates'
    iou_key = f"iou ({target_class_name})"
    dice_key = f"dice ({target_class_name})"
    val_iou_history_key = f"val_{iou_key}"
    val_dice_history_key = f"val_{dice_key}"

    # --- Configuration Printouts ---
    print(f"--- {target_class_name} Segmentation (IDRID) - {ENCODER} + Attention @ {IMAGE_SIZE}x{IMAGE_SIZE} (Balanced Patch Training + Online CLAHE) ---")
    print(f"Patch Size: {PATCH_SIZE}x{PATCH_SIZE}, Stride: {STRIDE}") # Now 384x384
    print(f"Using device: {DEVICE}")
    print(f"Class: {target_class_name} (Num classes: {NUM_CLASSES})")
    print(f"Mask Subdir: '{MASK_SUBDIR_NAMES[target_class_name]}'")
    print(f"Model input size: {IMAGE_SIZE}x{IMAGE_SIZE}") # Now 384x384
    print(f"Batch size: {BATCH_SIZE}") # Adjusted
    print(f"Epochs: {NUM_EPOCHS}")
    print(f"Learning Rate: {LEARNING_RATE}")
    print(f"Encoder: {ENCODER}")
    print(f"Decoder Attention Gates: True")
    print(f"Loss: BCE + Dice (Weights: {BCE_WEIGHT}/{DICE_WEIGHT})")
    print(f"Online Augmentation includes: CLAHE, Elastic, GridDist, Brightness/Contrast, ColorJitter, Noise, Blur, Dropout")
    print(f"Scheduler: CosineAnnealingLR")
    print(f"Saving results for: {MODEL_NAME}")
    print(f"----------------------------------------------------")

    # --- Define Patch Output Dirs (from Block 2) ---
    patch_save_mask_dir_specific = os.path.join(PATCH_SAVE_MASK_DIR_BASE, target_subdir_name)
    print(f"Balanced Patch Images Dir: {PATCH_SAVE_IMG_DIR}")
    print(f"Balanced Patch Masks Dir: {patch_save_mask_dir_specific}")
    print(f"----------------------------------------------------")

    # --- STEP 1: Generate Balanced Patches (or skip if exists) ---
    # <<< START ADDED CHECK >>>
    skip_generation = False
    if os.path.isdir(PATCH_SAVE_IMG_DIR):
        # Check if the directory is not empty (list first few potential files)
        potential_files = glob.glob(os.path.join(PATCH_SAVE_IMG_DIR, "*.jpg")) + \
                          glob.glob(os.path.join(PATCH_SAVE_IMG_DIR, "*.png")) + \
                          glob.glob(os.path.join(PATCH_SAVE_IMG_DIR, "*.tif"))
        if potential_files:
             print(f"STEP 1: Found existing balanced patches in {PATCH_SAVE_IMG_DIR}. Skipping generation.")
             skip_generation = True
        else:
             print(f"STEP 1: Directory {PATCH_SAVE_IMG_DIR} exists but is empty. Generating patches...")
    else:
         print(f"STEP 1: Directory {PATCH_SAVE_IMG_DIR} not found. Generating patches...")

    if not skip_generation:
        num_balanced_patches = generate_balanced_patches(
            image_dir=TRAIN_IMG_DIR,
            mask_base_dir=TRAIN_MASK_BASE_DIR,
            target_class_name=target_class_name,
            target_subdir_name=target_subdir_name,
            patch_save_img_dir=PATCH_SAVE_IMG_DIR,
            patch_save_mask_dir_base=PATCH_SAVE_MASK_DIR_BASE,
            patch_size=PATCH_SIZE,
            stride=STRIDE
        )
        if num_balanced_patches == 0:
            raise RuntimeError("Failed to generate any balanced patches. Check input data and patch generation logs.")
        print(f"\nFinished generating {num_balanced_patches} balanced patches.")
    # <<< END ADDED CHECK >>>
    print(f"----------------------------------------------------")

    # --- STEP 2: (REMOVED) Offline Augmentation ---
    print("STEP 2: SKIPPED Offline geometric augmentation.")
    print(f"----------------------------------------------------")


    # --- STEP 3: Prepare Data Entries from BALANCED Patches ---
    print(f"STEP 3: Preparing {target_class_name} data entries from BALANCED patches...")
    # Data entries always read from the balanced patch directory now
    DATA_ENTRY_IMG_DIR = PATCH_SAVE_IMG_DIR
    DATA_ENTRY_MASK_BASE = PATCH_SAVE_MASK_DIR_BASE

    train_val_entries = create_data_entries(DATA_ENTRY_IMG_DIR, DATA_ENTRY_MASK_BASE)
    # Add a check here if entries are empty, even if directory existed
    if not train_val_entries:
        if skip_generation:
             raise ValueError("FATAL: Patch generation was skipped, but failed to create data entries from existing patches. Check directory contents and filenames.")
        else:
             raise ValueError("FATAL: Patch generation ran, but failed to create data entries. Check generation logs and created files.")

    print(f"\nTotal {target_class_name} BALANCED patch samples found for training/validation: {len(train_val_entries)}\n")

    # --- Data Split ---
    print("--- Splitting balanced patch data ---")
    if len(train_val_entries) < 5:
        print("WARNING: Very few patch samples found. Using minimal validation set.")
        val_indices = [len(train_val_entries)-1] if len(train_val_entries) > 0 else []
        train_indices = list(range(len(train_val_entries)-1)) if len(train_val_entries) > 1 else []
        if not train_indices and len(train_val_entries) == 1: train_indices = [0]; val_indices = [0]
        elif not train_indices and not val_indices: print("ERROR: No entries to split.")
    else:
        train_indices, val_indices = train_test_split( list(range(len(train_val_entries))), test_size=0.2, random_state=42, shuffle=True )

    if not train_indices and not val_indices and len(train_val_entries)>0:
         print("WARN: Data split resulted in empty train/val lists, using all data for training.")
         train_indices = list(range(len(train_val_entries)))
         val_indices = []

    train_entries = [train_val_entries[i] for i in train_indices]
    val_entries = [train_val_entries[i] for i in val_indices]
    print(f"Balanced Patch data split: {len(train_entries)} train, {len(val_entries)} val samples.")

    # --- Create Datasets & DataLoaders ---
    print("Creating Balanced Patch Datasets...")
    train_dataset = IdridDataset(image_paths=[e['image_path'] for e in train_entries], mask_dict_list=[e['mask_paths'] for e in train_entries], class_names=CLASS_NAMES, transform=get_train_transforms(img_size=IMAGE_SIZE), image_size=IMAGE_SIZE ) if train_entries else None
    val_dataset = IdridDataset(image_paths=[e['image_path'] for e in val_entries], mask_dict_list=[e['mask_paths'] for e in val_entries], class_names=CLASS_NAMES, transform=get_val_transforms(img_size=IMAGE_SIZE), image_size=IMAGE_SIZE ) if val_entries else None
    print("Creating Balanced Patch DataLoaders...")
    # Ensure drop_last=True only if train_dataset is large enough
    drop_last_train = len(train_dataset) >= BATCH_SIZE if train_dataset else False
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, drop_last=drop_last_train) if train_dataset else None
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True) if val_dataset else None

    if train_loader: print(f"Train loader: {len(train_loader)} batches (Size: {BATCH_SIZE}, Balanced Patch Samples: {len(train_dataset)})")
    else: print("Train loader is empty or None.")
    if val_loader: print(f"Val loader: {len(val_loader)} batches (Size: {BATCH_SIZE}, Balanced Patch Samples: {len(val_dataset)})")
    else: print("Val loader is empty or None.")
    print(f"----------------------------------------------------")

    # --- STEP 4: Initialize Model, Loss, Optimizer, Scheduler ---
    print(f"STEP 4: Initializing Model ({ENCODER} + Attention - Input {IMAGE_SIZE}x{IMAGE_SIZE}), Loss, Optimizer, Scheduler...")
    # <<< Ensure IMAGE_SIZE is 384 when calling build_model >>>
    model = build_model(encoder=ENCODER, encoder_weights=ENCODER_WEIGHTS, classes=NUM_CLASSES, use_attention=True).to(DEVICE)
    criterion = BCEDiceLoss(weight_bce=BCE_WEIGHT, weight_dice=DICE_WEIGHT).to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-7)
    print(f"Using CosineAnnealingLR scheduler: T_max={NUM_EPOCHS}, eta_min=1e-7")
    print(f"----------------------------------------------------")

    # --- STEP 5: Load Checkpoint if available ---
    print(f"STEP 5: Attempting to load checkpoint: '{BEST_CHECKPOINT_PATH}'")
    start_epoch, best_val_iou_he, history = load_checkpoint(model, optimizer, scheduler, filename=BEST_CHECKPOINT_PATH)
    patience_counter = 0
    if start_epoch > 0 and os.path.isfile(BEST_CHECKPOINT_PATH):
        print(f"Adjusting scheduler to epoch {start_epoch}...")
        try:
            checkpoint_data = torch.load(BEST_CHECKPOINT_PATH, map_location='cpu')
            if 'scheduler_state_dict' in checkpoint_data:
                 # Need to load state THEN set last_epoch for CosineAnnealingLR
                 scheduler.load_state_dict(checkpoint_data['scheduler_state_dict'])
                 # Manually adjust internal state if needed, setting last_epoch directly might not be enough for all schedulers
                 # For CosineAnnealingLR, setting last_epoch should generally work if done after loading state.
                 scheduler.last_epoch = start_epoch # Set to epoch number completed
                 print(f"Scheduler state loaded and set to epoch {scheduler.last_epoch}")
            else:
                 print("WARN: Scheduler state not found in checkpoint, starting LR schedule from scratch.")
        except Exception as e_sched_load:
             print(f"WARN: Failed to load or set scheduler state: {e_sched_load}. Starting LR schedule from scratch.")


    # --- STEP 6: Training Loop ---
    print(f"STEP 6: Starting training from epoch {start_epoch} for {NUM_EPOCHS} epochs...")
    last_epoch_completed = start_epoch - 1

    if not train_loader:
         print("ERROR: Cannot start training loop, train_loader is empty/None.")
    # <<< Added check for val_loader as well >>>
    elif not val_loader:
         print("ERROR: Cannot start training loop, val_loader is empty/None. Validation is required for saving best model.")
    else:
        for epoch in range(start_epoch, NUM_EPOCHS):
            epoch_num = epoch + 1
            print(f"\n===== Epoch {epoch_num}/{NUM_EPOCHS} =====")
            start_time = time.time()

            # --- Train ---
            train_loss, _ = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)

            # --- Validate ---
            visualize_this_epoch = (epoch_num % 10 == 0) or (epoch_num == NUM_EPOCHS)
            val_loss, val_metrics = validate_one_epoch( model, val_loader, criterion, DEVICE, epoch, NUM_EPOCHS, visualize=visualize_this_epoch, vis_dir=VIS_DIR, num_vis_samples=2 )

            # --- Update History ---
            history['train_loss'].append(train_loss)
            history['val_loss'].append(val_loss)
            current_val_iou = val_metrics.get(iou_key, 0.0)
            current_val_dice = val_metrics.get(dice_key, 0.0)
            history.setdefault(val_iou_history_key, []).append(current_val_iou)
            history.setdefault(val_dice_history_key, []).append(current_val_dice)

            # --- Print Epoch Summary ---
            end_time = time.time(); epoch_duration = end_time - start_time
            # <<< Get LR *after* scheduler step for correct logging >>>
            # current_lr = optimizer.param_groups[0]['lr'] # Moved after scheduler.step()

            # --- Scheduling ---
            # Step the scheduler *after* the epoch completes (standard for CosineAnnealingLR)
            scheduler.step()
            current_lr = optimizer.param_groups[0]['lr'] # Get LR *after* step for logging

            print(f"Epoch {epoch_num} Summary | Duration: {epoch_duration:.2f}s | LR: {current_lr:.1e}")
            print(f"  Train Loss: {train_loss:.4f}")
            print(f"  Val Loss  : {val_loss:.4f} | Val {iou_key}: {current_val_iou:.4f} | Val {dice_key}: {current_val_dice:.4f}")


            # --- Saving Best Model ---
            is_best = current_val_iou > best_val_iou_he
            if is_best:
                print(f"  New best validation {iou_key}: {current_val_iou:.4f} (Prev: {best_val_iou_he:.4f})")
                best_val_iou_he = current_val_iou
                patience_counter = 0
                save_checkpoint({
                    'epoch': epoch, # Save epoch completed
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'scheduler_state_dict': scheduler.state_dict(), # Save state *after* step
                    'scheduler_type': type(scheduler).__name__,
                    'best_metric': best_val_iou_he, 'history': history, 'class_names': CLASS_NAMES,
                    'image_size': IMAGE_SIZE, 'patch_size': PATCH_SIZE, 'stride': STRIDE,
                    'encoder': ENCODER, 'batch_size': BATCH_SIZE, 'use_attention': True
                }, BEST_CHECKPOINT_PATH)
            else:
                patience_counter += 1

            # --- Early Stopping ---
            if patience_counter >= PATIENCE:
                print(f"\nEarly stopping triggered at epoch {epoch_num}.")
                break
            last_epoch_completed = epoch

    # --- Training Loop Finished ---
    print(f"\n--- Training Finished ({target_class_name} Model - Balanced Patch Run + Online CLAHE) ---")
    print(f"Completed {last_epoch_completed + 1} epoch(s).")
    print(f"Best Validation {iou_key} achieved: {best_val_iou_he:.4f}")
    print(f"Saving final model state to {FINAL_MODEL_PATH}")
    # Save the state dict of the model as it was at the END of the last completed epoch
    torch.save(model.state_dict(), FINAL_MODEL_PATH)

    # Plot history accumulated over the run
    plot_training_curves(history, save_path=TRAINING_PLOTS_PATH)

    print(f"\n--- Final Summary ({target_class_name} Model - Balanced Patch Run + Online CLAHE) ---")
    print(f"Model: {MODEL_NAME}")
    print(f"Best Validation {iou_key}: {best_val_iou_he:.4f}")
    print(f"Plots saved to: {TRAINING_PLOTS_PATH}")
    print(f"Best checkpoint saved to: {BEST_CHECKPOINT_PATH}")
    print(f"Final model saved to: {FINAL_MODEL_PATH}")
    print(f"Visualizations saved in: {VIS_DIR}")
    print(f"Balanced patches used for training saved in: {PATCH_SAVE_DIR_BASE}")
    print("Done.")

Using Original Train Img Path: /content/drive/MyDrive/IDRID/A.%20Segmentation/A. Segmentation/1. Original Images/a. Training Set
Using Original Train Mask Base Path: /content/drive/MyDrive/IDRID/A.%20Segmentation/A. Segmentation/2. All Segmentation Groundtruths/a. Training Set
MODEL_NAME set to: idrid_hard_exudates_256_attn_swin_s_w16_patch_fulltrain_patch
Saving checkpoints to: /content/drive/MyDrive/segmentation/idrid_hard_exudates_256_attn_swin_s_w16_patch_fulltrain_patch_best_model.pth
Preprocessing and Augmentation functions defined correctly (Patching only + Online CLAHE).
--- EX Segmentation (IDRID) - swinv2_small_window16_256 + Attention @ 256x256 (Balanced Patch Training + Online CLAHE) ---
Patch Size: 256x256, Stride: 128
Using device: cuda
Class: EX (Num classes: 1)
Mask Subdir: '3. Hard Exudates'
Model input size: 256x256
Batch size: 64
Epochs: 500
Learning Rate: 5e-05
Encoder: swinv2_small_window16_256
Decoder Attention Gates: True
Loss: BCE + Dice (Weights: 0.5/0.5)
Onlin

Pairing EX mask patches:   0%|          | 0/13360 [00:00<?, ?it/s]


--- Mask Patch File Check Report ---
  All 13360 expected 'EX' mask patch files were found and paired.
------------------------------

Total valid data entries created from balanced patches: 13360
--- End Creating Data Entries ---


Total EX BALANCED patch samples found for training/validation: 13360

--- Splitting balanced patch data ---
Balanced Patch data split: 10688 train, 2672 val samples.
Creating Balanced Patch Datasets...
--- Using online transforms for patches (Size: 256x256) with ONLINE CLAHE ---
Initialized Dataset with 10688 patch samples for class 'EX'.
Expecting patches of size 256x256.
--- Using validation transforms for patches (Size: 256x256) with ONLINE CLAHE ---
Initialized Dataset with 2672 patch samples for class 'EX'.
Expecting patches of size 256x256.
Creating Balanced Patch DataLoaders...
Train loader: 167 batches (Size: 64, Balanced Patch Samples: 10688)
Val loader: 42 batches (Size: 64, Balanced Patch Samples: 2672)
------------------------------------------

Train:   0%|          | 0/167 [00:00<?, ?it/s]

Val:   0%|          | 0/42 [00:00<?, ?it/s]

...

Train:   0%|          | 0/167 [00:00<?, ?it/s]

Val:   0%|          | 0/42 [00:00<?, ?it/s]

Epoch 149 Summary | Duration: 128.43s | LR: 4.0e-05
  Train Loss: 0.0598
  Val Loss  : 0.0357 | Val iou (EX): 0.8884 | Val dice (EX): 0.9206

===== Epoch 150/500 =====


Train:   0%|          | 0/167 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [6]:
pip install -q kaggle

In [7]:
from google.colab import files
files.upload()

Saving kaggle (1).json to kaggle (1).json


{'kaggle (1).json': b'{"username":"syedjaleel","key":"0df69e8cf10ab60c358c4c3d9dc0cef0"}'}

In [8]:
!mkdir ~/.kaggle

In [9]:
!cp kaggle(1).json ~/.kaggle/

/bin/bash: -c: line 1: syntax error near unexpected token `('
/bin/bash: -c: line 1: `cp kaggle(1).json ~/.kaggle/'


In [1]:
pip install segmentation_models_pytorch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 122.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 98.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 57.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 108.5 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninst

In [ ]:
# -*- coding: utf-8 -*-

# Block 0: Installs (Run this cell first in Colab)
# Ensure you have selected a Runtime with an A100 GPU
!pip install segmentation-models-pytorch timm albumentations --quiet
!pip install opencv-python-headless --quiet # Often preferred over standard opencv-python in headless envs
print("Required libraries installed.")
# Check GPU Allocation
!nvidia-smi

Required libraries installed.
Mon May  5 08:34:55 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------